In [83]:
import numpy as np
import pandas as pd
import yfinance as yf
import cvxpy as cp
import dimod
import neal
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from datetime import datetime
from typing import List, Tuple
from sklearn.covariance import LedoitWolf
from sklearn.cluster import AgglomerativeClustering
import os
import warnings
import hashlib
import itertools
import logging
import ast
import gc

warnings.filterwarnings('ignore')


# Parameters

In [84]:
MARKET_TICKERS = {
    "india": [
        "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS",
        "HINDUNILVR.NS", "ITC.NS", "SBIN.NS", "BHARTIARTL.NS", "LT.NS",
        "KOTAKBANK.NS", "AXISBANK.NS", "BAJFINANCE.NS", "ASIANPAINT.NS", "MARUTI.NS",
        "HCLTECH.NS", "SUNPHARMA.NS", "WIPRO.NS", "ULTRACEMCO.NS", "TITAN.NS",
        "TECHM.NS", "NESTLEIND.NS", "POWERGRID.NS", "NTPC.NS", "ONGC.NS",
        "JSWSTEEL.NS", "TATASTEEL.NS", "HINDALCO.NS", "COALINDIA.NS", "DIVISLAB.NS",
        "DRREDDY.NS", "CIPLA.NS", "ADANIENT.NS", "ADANIPORTS.NS", "BAJAJFINSV.NS",
        "BAJAJ-AUTO.NS", "HEROMOTOCO.NS", "EICHERMOT.NS", "TATACONSUM.NS", "BRITANNIA.NS",
        "GRASIM.NS", "INDUSINDBK.NS", "M&M.NS", "SBILIFE.NS", "HDFCLIFE.NS",
        "PIDILITIND.NS", "HAVELLS.NS", "SIEMENS.NS", "ABB.NS", "BERGEPAINT.NS",
    ],
    "usa": [
        "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "BRK-B", "UNH", "JNJ",
        "JPM", "V", "PG", "XOM", "HD", "CVX", "MRK", "ABBV", "PEP", "KO",
        "LLY", "AVGO", "COST", "MCD", "CSCO", "TMO", "ACN", "ABT", "DHR", "NEE",
        "TXN", "WMT", "PM", "QCOM", "HON", "UPS", "AMGN", "IBM", "SBUX", "GE",
        "MMM", "CAT", "BA", "GS", "MS", "BLK", "SPGI", "AXP", "RTX", "LMT",
    ],
    "europe": [
        "ASML.AS", "NESN.SW", "NOVN.SW", "ROG.SW",
        "SAP.DE", "SIE.DE", "ALV.DE", "BAS.DE", "BAYN.DE",
        "AIR.PA", "BNP.PA", "MC.PA", "SAN.PA", "OR.PA",
        "SHELL.AS", "HEIA.AS", "PHIA.AS", "UNA.AS",
        "AZN.L", "HSBA.L", "ULVR.L", "GSK.L", "RIO.L",
        "ENI.MI", "ENEL.MI", "ISP.MI", "UCG.MI",
        "IBE.MC", "SAN.MC", "BBVA.MC", "ITX.MC", "REP.MC",
        "NOVO-B.CO", "DSV.CO", "ORSTED.CO", "NESTE.HE", "NOKIA.HE",
        "ERIC-B.ST", "VOLV-B.ST", "ATCO-A.ST", "ABB.ST", "SEB-A.ST",
        "LONN.SW", "GIVN.SW", "GEBN.SW", "CFR.SW", "SREN.SW",
        "ADYEN.AS", "MT.AS", "UCB.BR",
    ],
    "japan": [
        "7203.T", "6758.T", "9984.T", "6861.T", "8306.T",
        "7974.T", "4063.T", "6098.T", "9433.T", "8035.T",
        "6954.T", "2914.T", "4519.T", "7751.T", "6301.T",
        "5108.T", "8801.T", "9022.T", "3382.T", "4543.T",
        "7267.T", "6902.T", "4661.T", "8411.T", "8316.T",
        "6702.T", "4502.T", "7832.T", "6501.T", "6503.T",
        "2802.T", "8766.T", "6273.T", "4523.T", "7011.T",
        "8053.T", "2501.T", "5401.T", "3407.T", "4568.T",
        "9432.T", "8031.T", "7733.T", "6645.T", "7201.T",
        "4901.T", "8002.T", "6971.T", "1925.T", "9020.T",
    ],
    "crypto": [
        "BTC-USD", "ETH-USD", "BNB-USD", "SOL-USD", "XRP-USD",
        "ADA-USD", "AVAX-USD", "DOGE-USD", "DOT-USD", "LINK-USD",
        "LRC-USD", "LTC-USD", "BCH-USD", "ATOM-USD", "1INCH-USD",
        "XLM-USD", "ALGO-USD", "VET-USD", "ICP-USD", "FIL-USD",
        "AAVE-USD", "EOS-USD", "XTZ-USD", "SAND-USD", "MANA-USD",
        "THETA-USD", "AXS-USD", "EGLD-USD", "ARB-USD", "NEAR-USD",
        "CELO-USD", "ENJ-USD", "CHZ-USD", "BAT-USD", "ZIL-USD",
        "WAVES-USD", "ICX-USD", "ZEC-USD", "DASH-USD", "XEM-USD",
        "QTUM-USD", "ONT-USD", "ZRX-USD", "BAL-USD", "CRV-USD",
        "SUSHI-USD", "YFI-USD", "LDO-USD", "MKR-USD", "SNX-USD",
    ],
}

MARKET_BENCHMARKS = {
    "india":  "^NSEI",
    "usa":    "^GSPC",
    "europe": "^STOXX50E",
    "japan":  "^N225",
    "crypto": "BTC-USD",
}


In [85]:
RISK_FREE_RATES = {
    "india":  0.065,
    "usa":    0.053,
    "europe": 0.039,
    "japan":  0.001,
    "crypto": 0.000,
}

MARKET_FR_WINDOW = {
    "india":  21,
    "usa":    14,
    "europe": 14,
    "japan":  21,
    "crypto": 7,
}

MARKET_REBALANCE_DAYS = {
    "india":  63,
    "usa":    63,
    "europe": 42,
    "japan":  42,
    "crypto": 21,
}

MARKET_TRANSACTION_COST = {
    "india":  0.006,
    "usa":    0.002,
    "europe": 0.003,
    "japan":  0.002,
    "crypto": 0.004,
}

MARKET_MAX_WEIGHT = {
    "india":  0.40,
    "usa":    0.35,
    "europe": 0.35,
    "japan":  0.40,
    "crypto": 0.25,
}

MARKET_MAX_VOL = {
    "india":  0.75,
    "usa":    0.60,
    "europe": 0.60,
    "japan":  0.70,
    "crypto": 2.00,
}

MARKET_MAX_CORRELATION = {
    "india":  0.85,
    "usa":    0.80,
    "europe": 0.80,
    "japan":  0.85,
    "crypto": 0.75,
}

MIN_DIVERSIFICATION_RATIO = {
    "india":  1.05,
    "usa":    1.10,
    "europe": 1.10,
    "japan":  1.05,
    "crypto": 1.15,
}

In [86]:
'''

MARKETS        = ["india", "usa", "europe", "japan", "crypto"]
UNIVERSE_SIZES = {"u10": 10, "u20": 20, "u30": 30, "u40": 40, "u50": 50}
CARDINALITIES  = [0.2, 0.3, 0.4]
LAMBDA_ALPHAS  = [2.0]
LOOKBACKS      = [126, 252, 504]
START_DATE     = "2022-01-01"
END_DATE       = "2026-01-01"
BACKTEST_DAYS  = 252
NUM_READS      = 1000
NUM_SWEEPS     = 3000
SA_SEEDS       = 5
PLOTS_DIR      = "plots/experiment_qubo_final/"

'''

'\n\nMARKETS        = ["india", "usa", "europe", "japan", "crypto"]\nUNIVERSE_SIZES = {"u10": 10, "u20": 20, "u30": 30, "u40": 40, "u50": 50}\nCARDINALITIES  = [0.2, 0.3, 0.4]\nLAMBDA_ALPHAS  = [2.0]\nLOOKBACKS      = [126, 252, 504]\nSTART_DATE     = "2022-01-01"\nEND_DATE       = "2026-01-01"\nBACKTEST_DAYS  = 252\nNUM_READS      = 1000\nNUM_SWEEPS     = 3000\nSA_SEEDS       = 5\nPLOTS_DIR      = "plots/experiment_qubo_final/"\n\n'

In [87]:
MARKETS        = ["india"]
UNIVERSE_SIZES = {"u40": 40, "u50": 50}
CARDINALITIES  = [0.2, 0.3, 0.4]
LAMBDA_ALPHAS  = [0.5]
LOOKBACKS      = [126, 252, 504]
START_DATE     = "2022-01-01"
END_DATE       = "2026-01-01"
BACKTEST_DAYS  = 252
NUM_READS      = 1000
NUM_SWEEPS     = 3000
SA_SEEDS       = 5
PLOTS_DIR      = "plots/experiment_qubo_fixed/"

# Logger

In [88]:
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

logger = logging.getLogger('qubo_optimization')
logger.setLevel(logging.DEBUG)
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)
logger.propagate = False

log_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file      = os.path.join(log_dir, f"qubo_optimization_{log_timestamp}.log")

console_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
file_formatter    = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(console_formatter)

file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(file_formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)
logger.info(f"Logging initialized — Console: INFO+, File: DEBUG+ ({log_file})")


15:16:23 - INFO - Logging initialized — Console: INFO+, File: DEBUG+ (logs\qubo_optimization_20260421_151623.log)


# Core Functions

## Data Layer

In [89]:
def download_data(market, start_date, end_date, cache_dir="data_cache"):
    if market not in MARKET_TICKERS:
        raise ValueError(f"Unknown market: '{market}'. Valid: {list(MARKET_TICKERS)}")

    os.makedirs(cache_dir, exist_ok=True)
    key          = hashlib.md5(f"{market}_{start_date}_{end_date}".encode()).hexdigest()
    prices_cache = os.path.join(cache_dir, f"{market}_{key}_prices.csv")
    volume_cache = os.path.join(cache_dir, f"{market}_{key}_volume.csv")

    if os.path.exists(prices_cache) and os.path.exists(volume_cache):
        try:
            prices = pd.read_csv(prices_cache, index_col=0, parse_dates=True)
            volume = pd.read_csv(volume_cache, index_col=0, parse_dates=True)
            logger.info(f"Loaded {market} from cache: {prices.shape}")
            return prices, volume
        except Exception as e:
            logger.warning(f"Cache read failed ({e}), re-downloading...")
            os.remove(prices_cache); os.remove(volume_cache)

    tickers = MARKET_TICKERS[market]
    raw     = yf.download(tickers, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)
    if raw.empty:
        raise ValueError(f"No data returned for market: {market}")

    prices = raw["Close"]  if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
    volume = raw["Volume"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Volume"]]
    common = prices.columns.intersection(volume.columns)
    prices, volume = prices[common], volume[common]

    original = set(prices.columns)
    prices   = prices.dropna(axis=1, thresh=int(0.8 * len(prices)))
    volume   = volume[prices.columns]
    prices   = prices.dropna()
    volume   = volume.loc[prices.index]

    dropped = original - set(prices.columns)
    if dropped:
        logger.warning(f"{market}: dropped {len(dropped)} tickers: {sorted(dropped)}")
    logger.info(f"{market}: {len(prices.columns)} tickers, {len(prices)} trading days")

    prices.to_csv(prices_cache); volume.to_csv(volume_cache)
    return prices, volume


In [90]:
def download_benchmark_data(market, start_date, end_date, cache_dir="data_cache"):
    if market not in MARKET_BENCHMARKS:
        raise ValueError(f"Unknown market: '{market}'. Valid: {list(MARKET_BENCHMARKS)}")

    os.makedirs(cache_dir, exist_ok=True)
    ticker     = MARKET_BENCHMARKS[market]
    key        = hashlib.md5(f"bench_{market}_{ticker}_{start_date}_{end_date}".encode()).hexdigest()
    cache_path = os.path.join(cache_dir, f"bench_{market}_{key}.csv")

    if os.path.exists(cache_path):
        try:
            df    = pd.read_csv(cache_path, index_col=0, parse_dates=True)
            close = df.iloc[:, 0] if df.shape[1] == 1 else df.squeeze()
            logger.info(f"Loaded benchmark {ticker} from cache: {len(close)} rows")
            return close
        except Exception as e:
            logger.warning(f"Benchmark cache read failed ({e}), re-downloading...")
            os.remove(cache_path)

    raw = yf.download(ticker, start=start_date, end=end_date,
                      auto_adjust=True, progress=False)
    if raw.empty:
        raise ValueError(f"No benchmark data for {market} ({ticker})")

    close = raw["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    close = close.dropna(); close.name = ticker

    logger.info(f"Downloaded benchmark {ticker}: {len(close)} rows")
    close.to_csv(cache_path)
    return close


## Universe Construction

In [91]:
def _filter_liquidity(prices, volume, min_pct=0.95):
    mask    = (volume > 0).sum() >= min_pct * len(prices)
    dropped = (~mask).sum()
    if dropped:
        logger.warning(f"Liquidity filter dropped {dropped} tickers")
    return prices.loc[:, mask], volume.loc[:, mask]


def _filter_adtv(prices, volume, size):
    adtv = (prices * volume).mean()
    top  = adtv.nlargest(min(size, len(adtv))).index
    return prices[top], volume[top]


def _filter_volatility(prices, volume, lookback, max_vol):
    ann_vol = prices.iloc[-lookback:].pct_change().dropna().std() * np.sqrt(252)
    mask    = ann_vol <= max_vol
    dropped = (~mask).sum()
    if dropped:
        logger.warning(f"Volatility filter dropped {dropped} tickers (max={max_vol:.0%})")
    return prices.loc[:, mask], volume.loc[:, mask]


def _cluster_select(prices, volume, lookback, n_clusters, max_correlation: float = 0.85):
    if prices.shape[1] <= n_clusters:
        return prices
    returns = prices.iloc[-lookback:].pct_change().dropna()
    dist    = np.clip(1 - np.abs(returns.corr().values), 0, 1)
    np.fill_diagonal(dist, 0)
    labels  = AgglomerativeClustering(
        n_clusters=n_clusters, metric="precomputed", linkage="average"
    ).fit_predict(dist)
    adtv   = (prices * volume).mean()
    chosen = [adtv[prices.columns[labels == c]].idxmax() for c in range(n_clusters)]

    # post-cluster correlation filter (Option 2)
    if len(chosen) > 1:
        ret_chosen  = returns[chosen]
        corr_matrix = ret_chosen.corr().values
        adtv_chosen = adtv[chosen].values
        keep        = np.ones(len(chosen), dtype=bool)
        for i in range(len(chosen)):
            if not keep[i]:
                continue
            for j in range(i + 1, len(chosen)):
                if keep[j] and corr_matrix[i, j] > max_correlation:
                    if adtv_chosen[i] >= adtv_chosen[j]:
                        keep[j] = False
                    else:
                        keep[i] = False
                        break
        chosen = [c for c, k in zip(chosen, keep) if k]
        if len(chosen) < n_clusters:
            logger.info(
                f"Correlation filter removed {n_clusters - len(chosen)} assets "
                f"(max_corr={max_correlation:.2f})"
            )

    logger.info(f"Cluster selection -> {len(chosen)} assets: {chosen}")
    return prices[chosen]


def build_universe(prices: pd.DataFrame, volume: pd.DataFrame, size: int,
                   lookback: int, market: str, pool_multiplier: int = 2,
                   max_correlation: float = None) -> pd.DataFrame:
    if prices.empty:
        raise ValueError("prices is empty")

    max_vol      = MARKET_MAX_VOL.get(market, 1.0)
    pool_size    = size * pool_multiplier
    if max_correlation is None:
        max_correlation = MARKET_MAX_CORRELATION.get(market, 0.85)

    logger.info(f"Building universe: market={market}, size={size}, "
                f"pool_size={pool_size}, lookback={lookback}, max_vol={max_vol:.0%}")

    prices, volume = _filter_liquidity(prices, volume)
    prices, volume = _filter_adtv(prices, volume, pool_size)
    prices, volume = _filter_volatility(prices, volume, lookback, max_vol)

    # FIX 2: cluster into pool_size candidates first, then trim to final size
    prices = _cluster_select(prices, volume, lookback, n_clusters=pool_size,
                             max_correlation=max_correlation)

    if prices.empty:
        raise ValueError(f"No assets survived universe construction for {market}")

    # trim to exactly `size` — rank by recent ADTV as tiebreaker
    if prices.shape[1] > size:
        recent         = volume[prices.columns].iloc[-lookback:]
        adtv           = recent.mean()
        top_tickers    = adtv.nlargest(size).index
        prices         = prices[top_tickers]
        logger.info(f"Trimmed to {size} assets by ADTV from {prices.shape[1] + len(adtv) - len(adtv)} candidates")

    if prices.empty:
        raise ValueError(f"No assets survived trimming for {market}")

    logger.info(f"Universe built: {prices.shape[1]} assets -> {list(prices.columns)}")
    return prices

## Statistics & Expected Returns

In [92]:
def compute_expected_returns(prices: pd.DataFrame, lookback: int) -> np.ndarray:
    """
    James-Stein shrinkage estimator for expected returns.
    Shrinks each asset mean toward the cross-sectional grand mean.
    kappa = min(1, N/T): stronger shrinkage when T is small relative to N.
    """
    returns   = prices.pct_change().dropna()
    mu_sample = returns.mean().values * 252
    n, T      = len(mu_sample), len(returns)
    grand     = mu_sample.mean()
    kappa     = min(1.0, n / T)
    return (1.0 - kappa) * mu_sample + kappa * grand


def compute_statistics(prices: pd.DataFrame, use_shrinkage: bool = True,
                       use_garch: bool = False):
    """
    Returns (mu, cov, returns) or (mu, cov, returns, garch_vols) when use_garch=True
    and GARCH succeeds.

    mu  -- James-Stein shrinkage estimator (annualised) when use_shrinkage=True,
           otherwise annualised sample mean.
    cov -- Layer-1 covariance (Ledoit-Wolf or PCA factor model), optionally
           updated by Layer-2 GARCH diagonal rescaling.

    Layer 1 (always runs): T/N guard.
      T/N >= 5 -> Ledoit-Wolf shrinkage.
      T/N <  5 -> PCA factor model: K components explaining 80 % variance,
                  K in [2, min(T//10, N//2, 10)].

    Layer 2 (opt-in, use_garch=False): GARCH(1,1) diagonal rescaling.
      Fits per-asset GARCH, extracts 1-step-ahead forecast vols, keeps Layer-1
      correlation structure, replaces diagonal vols with GARCH forecasts.
      Returns 4-tuple (mu, cov, returns, garch_vols) on success.

    Drops zero-variance columns.
    """
    from sklearn.decomposition import PCA

    returns = prices.pct_change().dropna()

    constant_cols = returns.columns[returns.std() < 1e-10]
    if len(constant_cols):
        logger.warning(f"Dropping {len(constant_cols)} zero-variance assets: {list(constant_cols)}")
        returns = returns.drop(columns=constant_cols)

    if returns.empty or returns.shape[1] == 0:
        raise ValueError("No valid assets remaining after filtering constant-price columns")

    T        = len(returns)
    N        = returns.shape[1]
    T_over_N = T / N

    # ── Layer 1: T/N guard ───────────────────────────────────────────────────
    if T_over_N < 5:
        K_cap = max(min(T // 10, N // 2, 10), 2)
        pca_full = PCA(n_components=K_cap)
        pca_full.fit(returns.values)
        cumvar = np.cumsum(pca_full.explained_variance_ratio_)
        K = int(np.searchsorted(cumvar, 0.80)) + 1
        K = max(min(K, K_cap), 2)

        pca_k   = PCA(n_components=K)
        factors = pca_k.fit_transform(returns.values)
        B_mat   = pca_k.components_.T

        factor_cov   = np.cov(factors.T) * 252
        residuals    = returns.values - factors @ B_mat.T
        idio_var     = np.var(residuals, axis=0) * 252

        cov      = B_mat @ factor_cov @ B_mat.T + np.diag(idio_var)
        explained = float(cumvar[K - 1])
        logger.info(
            f"T/N={T_over_N:.1f} < 5 -- PCA factor model activated, "
            f"K={K} factors explaining {explained:.1%} variance"
        )
    else:
        cov = LedoitWolf().fit(returns).covariance_ * 252

    # ── Layer 2: GARCH diagonal (optional) ──────────────────────────────────
    garch_vols_out = None
    if use_garch:
        try:
            from arch import arch_model

            diag_vols  = np.sqrt(np.diag(cov))
            garch_vols = diag_vols.copy()

            for i, col in enumerate(returns.columns):
                try:
                    am  = arch_model(returns[col] * 100, vol="Garch", p=1, q=1,
                                     rescale=False)
                    res = am.fit(disp="off")
                    fc  = res.forecast(horizon=1)
                    var_pct_daily = float(fc.variance.values[-1, 0])
                    garch_vols[i] = np.sqrt(var_pct_daily * 252) / 100.0
                except Exception as e:
                    logger.warning(
                        f"GARCH fitting failed for {col}, "
                        f"using Layer 1 vol ({diag_vols[i]:.4f}): {e}"
                    )

            corr      = cov / np.outer(diag_vols, diag_vols)
            D         = np.diag(garch_vols)
            cov       = D @ corr @ D
            sigma     = garch_vols
            garch_vols_out = garch_vols  # store for 4-tuple return
            logger.info(
                f"GARCH diagonal applied: vol range "
                f"[{sigma.min():.4f}, {sigma.max():.4f}]"
            )
        except Exception as e:
            logger.warning(
                f"GARCH layer failed entirely, using Layer 1 covariance: {e}"
            )

    # ── compute mu ────────────────────────────────────────────────────────────
    if use_shrinkage:
        mu = compute_expected_returns(prices.loc[returns.index[0]:], lookback=T)
    else:
        mu = returns.mean().values * 252

    logger.debug(
        f"Statistics: T/N={T_over_N:.1f}, "
        f"mu=[{mu.min():.4f}, {mu.max():.4f}], "
        f"vol=[{np.sqrt(np.diag(cov)).min():.4f}, {np.sqrt(np.diag(cov)).max():.4f}]"
    )

    if garch_vols_out is not None:
        return mu, cov, returns, garch_vols_out
    return mu, cov, returns


## Global Scaling Helper

In [93]:
def scale_inputs(mu: np.ndarray, cov: np.ndarray
                 ) -> Tuple[np.ndarray, np.ndarray, float, float]:
    """
    Normalise mu and cov so all QUBO coefficients are comparable in magnitude.

    mu_norm  = mu  / max(|mu|)
    cov_norm = cov / max(|cov|)

    Used consistently in build_qubo (Step 1), lambda computation (Step 5),
    and rebalancing — ensuring no double-normalisation arises anywhere.

    Returns (mu_norm, cov_norm, mu_scale, cov_scale).
    """
    mu_scale  = float(np.maximum(np.max(np.abs(mu)),  1e-10))
    cov_scale = float(np.maximum(np.max(np.abs(cov)), 1e-10))
    return mu / mu_scale, cov / cov_scale, mu_scale, cov_scale


## Lagrangian-Informed q Scaling

In [94]:
def compute_lagrangian_q(mu_norm: np.ndarray, cov_norm: np.ndarray,
                          eps: float = 1e-6) -> float:
    """
    Derive the risk-aversion coefficient q from the Lagrangian dual.

    Given the regularised Markowitz solution:
        w* = (cov_norm + eps*I)^{-1} mu_norm

    q is the ratio of portfolio return to portfolio variance:
        q = (mu_norm^T w*) / (w*^T cov_norm w* + eps)

    This replaces the heuristic effective_q >= 0.3 clamp from v3 with a
    theoretically grounded value that adapts to the specific mu/cov realisation.

    Numerical safeguards:
    - Diagonal regularisation eps*I prevents singular-matrix failures.
    - eps added to denominator avoids division-by-zero.
    - Result clipped to [0.3, 3.0]:
        lower bound: return signal always contributes to selection.
        upper bound: variance terms are never completely swamped.
    """
    n         = len(mu_norm)
    Sigma_reg = cov_norm + eps * np.eye(n)
    try:
        w_star = np.linalg.solve(Sigma_reg, mu_norm)
    except np.linalg.LinAlgError:
        w_star = np.linalg.lstsq(Sigma_reg, mu_norm, rcond=None)[0]

    numerator   = float(mu_norm @ w_star)
    denominator = float(w_star @ cov_norm @ w_star) + eps
    q_raw       = numerator / denominator
    q           = float(np.clip(q_raw, 0.3, 3.0))

    logger.debug(f"compute_lagrangian_q: q_raw={q_raw:.6f} -> clipped q={q:.4f}")
    return q


## QUBO Construction

In [95]:
def build_qubo(mu: np.ndarray, cov: np.ndarray,
               B: int, lambda_alpha: float = 2.0,
               cardinality_ratio: float = None,
               garch_vols: np.ndarray = None,
               ) -> Tuple[np.ndarray, Tuple[float, float], np.ndarray]:
    """
    Build the upper-triangular QUBO matrix Q. Fully vectorised -- no Python loops.

    Decision variables: x_i in {0,1}.  Cardinality constraint: sum(x_i) = B.
    QUBO performs asset *selection* only; weights are computed separately
    by CVXPY (optimize_weights).

    Pipeline
    --------
    Step 1 -- Normalise inputs via scale_inputs so coefficients are comparable.

    Step 2 -- Compute q via compute_lagrangian_q.

    Step 3 -- Build objective matrix (vectorised).

    Step 4 -- Save obj_scores = -diag(Q) BEFORE penalty is added.

    Step 5 -- Compute effective_alpha (dynamic lambda) and lambda_penalty.
              If garch_vols provided: uses vol dispersion to scale alpha.
              Cardinality boost (>=0.35) folded into effective_alpha.

    Step 6 -- Add budget-constraint expansion (sum x_i - B)^2 (vectorised).

    Returns
    -------
    Q          : (n,n) upper-triangular QUBO matrix ready for SA
    beta_range : (beta_min, beta_max) annealing schedule for neal SA
    obj_scores : (n,) pre-penalty asset quality scores
    """
    n = len(mu)

    # Step 1 -- normalise
    mu_norm, cov_norm, _, _ = scale_inputs(mu, cov)

    # Step 2 -- Lagrangian q
    q = compute_lagrangian_q(mu_norm, cov_norm)

    # Step 3 -- objective matrix (fully vectorised)
    Q = np.zeros((n, n))
    np.fill_diagonal(Q, q * np.diag(cov_norm) - mu_norm)
    i_idx, j_idx    = np.triu_indices(n, k=1)
    Q[i_idx, j_idx] = 2.0 * q * cov_norm[i_idx, j_idx]

    # Step 4 -- pre-penalty scores (before constraint terms distort diagonal)
    obj_scores = -np.diag(Q).copy()

    # beta_range for SA annealing schedule (computed on objective-only scale)
    obj_scale  = float(np.max(np.abs(Q)))
    beta_range = (0.5 / obj_scale, 10.0 / obj_scale) if obj_scale > 1e-10 else (0.1, 5.0)

    # Step 5 -- dynamic effective_alpha + lambda computation
    mean_vol = np.sqrt(np.diag(cov)).mean()  # original cov, not normalised

    if garch_vols is not None:
        dispersion      = garch_vols.std() / (garch_vols.mean() + 1e-6)
        effective_alpha = lambda_alpha * (1 + mean_vol) / (dispersion + 1e-6)
    else:
        effective_alpha = lambda_alpha * (1 + mean_vol)

    if cardinality_ratio is not None and cardinality_ratio >= 0.35:
        effective_alpha *= 1.5

    logger.info(
        f"dynamic lambda: base={lambda_alpha:.4f} -> effective={effective_alpha:.4f} "
        f"(mean_vol={mean_vol:.4f}, "
        f"{'garch_dispersion=' + f'{dispersion:.4f}' if garch_vols is not None else 'no GARCH'})"
    )

    objective_scale = max(float(np.max(np.abs(Q))), float(np.max(np.abs(mu_norm))))
    lambda_penalty  = effective_alpha * objective_scale

    # ── Layer 3: lambda sanity warnings ──────────────────────────────────────
    if lambda_penalty < 0.1 * objective_scale:
        logger.warning(
            f"WARNING -- lambda_alpha={lambda_alpha} too small: "
            f"penalty={lambda_penalty:.4f} may fail to enforce cardinality, "
            f"constraint violations likely"
        )
    if lambda_penalty > 10.0 * objective_scale:
        logger.warning(
            f"WARNING -- lambda_alpha={lambda_alpha} too large: "
            f"penalty={lambda_penalty:.4f} dominates objective by "
            f"{lambda_penalty / objective_scale:.1f}x, asset quality signal suppressed"
        )

    # Step 6 -- budget constraint (sum x_i - B)^2 expansion (vectorised)
    np.fill_diagonal(Q, np.diag(Q) + lambda_penalty * (1 - 2 * B))
    Q[i_idx, j_idx] += 2.0 * lambda_penalty

    logger.debug(
        f"build_qubo: n={n}, B={B}, q={q:.4f}, "
        f"objective_scale={objective_scale:.4f}, "
        f"effective_alpha={effective_alpha:.4f}, lambda={lambda_penalty:.4f}, "
        f"mean_vol={mean_vol:.4f}, garch={'yes' if garch_vols is not None else 'no'}"
    )
    return Q, beta_range, obj_scores


## Cardinality Enforcement

In [96]:
def enforce_cardinality(x: np.ndarray, scores: np.ndarray, B: int) -> np.ndarray:
    """
    Vectorised adjustment to meet cardinality exactly.
    Uses np.flatnonzero + np.argpartition (no Python loops).
    Guards against pool-too-small edge case with an explicit warning.
    """
    x_enforced     = x.copy()
    selected_idx   = np.flatnonzero(x_enforced)
    unselected_idx = np.flatnonzero(x_enforced == 0)
    num_selected   = len(selected_idx)

    if num_selected > B:
        n_drop = num_selected - B
        order  = np.argpartition(scores[selected_idx], n_drop)[:n_drop]
        x_enforced[selected_idx[order]] = 0

    elif num_selected < B:
        n_add = B - num_selected
        if len(unselected_idx) == 0:
            logger.warning(
                f"enforce_cardinality: need {n_add} more assets "
                f"but no unselected assets remain -- cardinality unfulfilled"
            )
        else:
            avail = min(n_add, len(unselected_idx))
            if avail < n_add:
                logger.warning(
                    f"enforce_cardinality: pool too small -- "
                    f"can only add {avail}, need {n_add}"
                )
            order = np.argpartition(scores[unselected_idx], -avail)[-avail:]
            x_enforced[unselected_idx[order]] = 1

    final_count = int(x_enforced.sum())
    if final_count != B:
        logger.warning(f"enforce_cardinality: post-enforcement count={final_count} != B={B}")

    return x_enforced


## QUBO Solver

In [97]:
def solve_qubo_sa(Q: np.ndarray, num_reads: int = 1000, B: int = 5,
                  seed: int = 42, beta_range: Tuple[float, float] = (0.1, 5.0),
                  n_seeds: int = 5, num_sweeps: int = 1000,
                  obj_scores: np.ndarray = None) -> Tuple[np.ndarray, float, int, int]:
    """
    Simulated-annealing QUBO solver (D-Wave neal).

    Runs n_seeds independent SA calls and picks the lowest-energy solution
    using vectorised einsum energy evaluation.
    BQM is built once and reused across all seed calls for efficiency.

    obj_scores: pre-penalty asset quality scores (from build_qubo).
                Used for cardinality enforcement tie-breaking.
                Falls back to -diag(Q) if None (penalty-contaminated).
    """
    n = Q.shape[0]

    # build BQM once -- reused across seed calls
    qubo_dict = {}
    ri, ci    = np.triu_indices(n, k=0)
    for i, j in zip(ri, ci):
        v = Q[i, j]
        if abs(v) > 1e-10:
            qubo_dict[(int(i), int(j))] = float(v)

    bqm     = dimod.BinaryQuadraticModel.from_qubo(qubo_dict)
    sampler = neal.SimulatedAnnealingSampler()
    Q_upper = np.triu(Q)

    best_energy   = np.inf
    best_solution = None

    for s in range(n_seeds):
        sampleset = sampler.sample(
            bqm, num_reads=num_reads,
            seed=seed + s, beta_range=list(beta_range), num_sweeps=num_sweeps,
        )
        # vectorised extraction shape (num_reads, n)
        samples_matrix = np.array(
            [[row.sample.get(i, 0) for i in range(n)]
             for row in sampleset.data(fields=["sample"])]
        )
        # vectorised energy via einsum: x^T Q_upper x for each sample
        energies = np.einsum("ij,jk,ik->i", samples_matrix, Q_upper, samples_matrix)
        idx      = int(np.argmin(energies))

        if energies[idx] < best_energy:
            best_energy   = float(energies[idx])
            best_solution = samples_matrix[idx].astype(int).copy()

    raw_count = int(best_solution.sum())
    logger.debug(
        f"solve_qubo_sa: best of {n_seeds} seeds | "
        f"energy={best_energy:.4f} | selected={raw_count} (target {B})"
    )

    violated = 0
    if raw_count != B:
        logger.debug(f"Cardinality violation: {raw_count} != {B}, enforcing...")
        scores        = obj_scores if obj_scores is not None else -np.diag(Q)
        best_solution = enforce_cardinality(best_solution, scores, B)
        violated      = 1

    final_energy = float(best_solution @ Q_upper @ best_solution)
    logger.debug(f"solve_qubo_sa: final={int(best_solution.sum())} assets, "
                 f"energy={final_energy:.4f}")

    return best_solution, final_energy, violated, raw_count


## Weight Optimisation

In [98]:
def optimize_weights(mu: np.ndarray, cov: np.ndarray, selected: np.ndarray,
                     max_weight: float = 0.40,
                     min_diversification_ratio: float = 1.0) -> np.ndarray:
    """
    Continuous mean-variance optimisation via CVXPY/ECOS.

    QUBO selects *which* assets to hold; this function allocates *how much*.
    risk_aversion is fixed at 0.5. Falls back to equal weights on solver failure.

    min_diversification_ratio: when > 1.0, adds a DR lower-bound constraint
    as a proper SOCP cone:
        ||chol(Σ) w||₂  <=  (σ_i · w) / dr_min
    which is DCP-compliant (norm <= affine) and handled natively by ECOS.
    If infeasible, retries without DR before falling back to equal weights.
    """
    n       = len(mu)
    indices = np.where(selected == 1)[0]
    k       = len(indices)

    if k == 0:
        logger.warning("No assets selected, returning zero weight vector")
        return np.zeros(n)

    mu_s          = mu[indices]
    cov_s         = cov[np.ix_(indices, indices)]
    risk_aversion = 0.5
    effective_max = max(max_weight, 1.0 / k)

    w           = cp.Variable(k)
    objective   = cp.Minimize(cp.quad_form(w, cov_s) - risk_aversion * mu_s @ w)
    constraints = [cp.sum(w) == 1, w >= 0, w <= effective_max]

    # DR constraint as SOCP: ||chol(Σ+εI) w||₂ <= (σ_i · w) / dr_min
    # Equivalent to DR = (w^T σ) / sqrt(w^T Σ w) >= dr_min, but DCP-compliant.
    if min_diversification_ratio > 1.0:
        individual_vols_s = np.sqrt(np.diag(cov_s))
        cov_s_reg = cov_s + 1e-8 * np.eye(k)
        try:
            chol_s = np.linalg.cholesky(cov_s_reg)
        except np.linalg.LinAlgError:
            # fallback: use diagonal sqrt if Cholesky fails
            chol_s = np.diag(np.sqrt(np.diag(cov_s_reg)))
        constraints += [
            cp.norm(chol_s @ w, 2) <= (individual_vols_s @ w) / min_diversification_ratio
        ]

    problem = cp.Problem(objective, constraints)

    try:
        problem.solve(solver=cp.ECOS, verbose=False)
        if problem.status in ("optimal", "optimal_inaccurate") and w.value is not None:
            w_val                 = np.maximum(w.value, 0)
            weights_full          = np.zeros(n)
            weights_full[indices] = w_val / w_val.sum()
            return weights_full

        if min_diversification_ratio > 1.0:
            logger.warning(
                f"optimize_weights: DR constraint infeasible "
                f"(dr_min={min_diversification_ratio:.2f}, status={problem.status}), "
                f"retrying without it"
            )
            w2           = cp.Variable(k)
            constraints2 = [cp.sum(w2) == 1, w2 >= 0, w2 <= effective_max]
            problem2     = cp.Problem(
                cp.Minimize(cp.quad_form(w2, cov_s) - risk_aversion * mu_s @ w2),
                constraints2
            )
            problem2.solve(solver=cp.ECOS, verbose=False)
            if problem2.status in ("optimal", "optimal_inaccurate") and w2.value is not None:
                w_val                 = np.maximum(w2.value, 0)
                weights_full          = np.zeros(n)
                weights_full[indices] = w_val / w_val.sum()
                return weights_full

        logger.warning(f"optimize_weights: status={problem.status}, "
                       f"k={k}, max_weight={effective_max:.2f} -- equal-weight fallback")

    except Exception as e:
        logger.warning(f"optimize_weights: solver failed ({type(e).__name__}: {e}) -- equal-weight fallback")

    weights_full          = np.zeros(n)
    weights_full[indices] = 1.0 / k
    return weights_full

## Metrics

In [99]:
def compute_metrics(weights: np.ndarray, mu: np.ndarray, cov: np.ndarray,
                    prices_test: pd.DataFrame, market: str = "usa") -> dict:
    """
    Ex-ante diagnostics (mu/cov from training window):
        expected_return, volatility, ex_ante_sharpe, diversification_ratio.
    Realised OOS metrics (held-out prices):
        ann_return, total_return, max_drawdown, realised_sharpe.
    rfr_adj_ann_return = ann_return - rfr  (cross-market comparable).

    sharpe_ratio key returns the ex-ante Sharpe for selection and reporting,
    consistent with the walk-forward pipeline which tracks last-period mu/cov.
    realised_sharpe is retained as a separate diagnostic field.
    """
    risk_free_rate = RISK_FREE_RATES.get(market, 0.0)
    rfr_daily      = risk_free_rate / 252.0

    # ex-ante
    expected_return = float(weights @ mu)
    variance        = float(weights @ cov @ weights)
    volatility      = float(np.sqrt(variance))
    ex_ante_sharpe  = ((expected_return - risk_free_rate) / volatility
                       if volatility > 1e-10 else 0.0)
    individual_vols       = np.sqrt(np.diag(cov))
    diversification_ratio = (float(weights @ individual_vols) / volatility
                             if volatility > 1e-10 else 0.0)

    # realised OOS (vectorised)
    returns_test = prices_test.pct_change().dropna()
    port_returns = returns_test.values @ weights
    excess_daily = port_returns - rfr_daily
    pv           = np.cumprod(1.0 + port_returns)

    total_return   = float(pv[-1] - 1.0)
    n_days         = len(pv)
    ann_return     = (float((1.0 + total_return) ** (252.0 / n_days) - 1.0)
                      if n_days > 0 else 0.0)
    mean_excess    = float(np.mean(excess_daily))
    std_excess     = float(np.std(excess_daily, ddof=1))
    realised_sharpe = (mean_excess / std_excess * np.sqrt(252)
                       if std_excess > 1e-10 else 0.0)
    rfr_adj_ann_return = ann_return - risk_free_rate

    cum_max      = np.maximum.accumulate(pv)
    max_drawdown = float(((pv - cum_max) / cum_max).min())

    logger.debug(
        f"compute_metrics [{market}]: "
        f"ex_ante_sharpe={ex_ante_sharpe:.4f} | "
        f"realised_sharpe={realised_sharpe:.4f} | "
        f"ann_return={ann_return:.4f} | max_dd={max_drawdown:.4f}"
    )

    return {
        # sharpe_ratio == ex-ante Sharpe (primary metric for selection/reporting)
        "sharpe_ratio":          ex_ante_sharpe,
        "ann_return":            ann_return,
        "rfr_adj_ann_return":    rfr_adj_ann_return,
        "total_return":          total_return,
        "max_drawdown":          max_drawdown,
        "ex_ante_sharpe":        ex_ante_sharpe,
        "realised_sharpe":       realised_sharpe,
        "expected_return":       expected_return,
        "volatility":            volatility,
        "diversification_ratio": diversification_ratio,
    }


# UNUSED — retained for external use
def compute_max_drawdown(portfolio_values: pd.Series) -> float:
    pv      = portfolio_values.values
    cum_max = np.maximum.accumulate(pv)
    return float(((pv - cum_max) / cum_max).min())


# UNUSED — retained for external use
def backtest_benchmark(benchmark_series: pd.Series, initial_capital: float) -> pd.Series:
    b = benchmark_series.dropna()
    return b / b.iloc[0] * initial_capital


# UNUSED — retained for external use
def compute_annualized_return(portfolio_values: pd.Series) -> float:
    pv           = portfolio_values.values
    total_return = pv[-1] / pv[0] - 1.0
    n_days       = len(pv)
    return float((1.0 + total_return) ** (252.0 / n_days) - 1.0) if n_days > 0 else 0.0

## Stability Analysis

In [100]:
def run_stability_analysis(Q: np.ndarray, B: int, n_seeds: int = 5,
                           num_reads: int = 1000, num_sweeps: int = 1000,
                           beta_range: Tuple[float, float] = (0.1, 5.0),
                           obj_scores: np.ndarray = None) -> dict:
    """
    Run n_seeds independent SA calls and return a majority-vote consensus.
    Passes obj_scores through to enforce_cardinality for informed tie-breaking.
    Aggregates violation count as a single summary line (not per-seed noise).
    """
    solutions, energies, violations = [], [], []

    for s in range(n_seeds):
        sol, energy, violated, _ = solve_qubo_sa(
            Q, num_reads=num_reads, B=B, seed=s,
            beta_range=beta_range, n_seeds=1, num_sweeps=num_sweeps,
            obj_scores=obj_scores,
        )
        solutions.append(sol)
        energies.append(energy)
        violations.append(violated)

    solutions_arr    = np.array(solutions)
    total_violations = int(sum(violations))
    if total_violations > 0:
        logger.info(f"Cardinality violations: {total_violations}/{n_seeds} seeds enforced")

    consensus = (solutions_arr.mean(axis=0) >= 0.5).astype(int)
    if int(consensus.sum()) != B:
        scores    = obj_scores if obj_scores is not None else -np.diag(Q)
        consensus = enforce_cardinality(consensus, scores, B)

    asset_consistency = float(np.mean([
        np.sum(solutions_arr[i] == consensus) / len(consensus)
        for i in range(n_seeds)
    ]))

    result = {
        "consensus_solution": consensus,
        "mean_energy":        float(np.mean(energies)),
        "std_energy":         float(np.std(energies)),
        "violation_rate":     float(np.mean(violations)),
        "asset_consistency":  asset_consistency,
    }
    logger.debug(f"Stability: mean_energy={result['mean_energy']:.4f}, "
                 f"std_energy={result['std_energy']:.4f}, "
                 f"violation_rate={result['violation_rate']:.3f}, "
                 f"consistency={asset_consistency:.3f}")
    return result


## Rebalancing & Backtesting

In [101]:
def rebalance_portfolio(prices: pd.DataFrame, tickers: List[str],
                        market: str,
                        initial_capital: float = 1_600_000,
                        B: int = 5,
                        lookback_days: int = 252,
                        lambda_alpha: float = 2.0,
                        min_diversification_ratio: float = 1.0) -> Tuple[pd.Series, List]:
    rebalance_days   = MARKET_REBALANCE_DAYS.get(market, 63)
    tc_rate          = MARKET_TRANSACTION_COST.get(market, 0.002)
    portfolio_values = []
    dates            = []
    rebalance_info   = []
    cash             = initial_capital
    shares           = np.zeros(len(prices.columns))

    logger.info(f"Starting rebalancing: market={market}, B={B}, "
                f"rebalance_days={rebalance_days}, capital={initial_capital:,.0f}")

    for i in range(lookback_days, len(prices), rebalance_days):
        rebalance_date = prices.index[i]
        logger.info(f"Rebalancing on {rebalance_date.date()}")

        train_window   = prices.iloc[i - lookback_days:i]
        result         = compute_statistics(train_window, use_shrinkage=True, use_garch=True)
        garch_vols     = result[3] if len(result) == 4 else None
        mu, cov, _     = result[:3]
        N              = len(mu)
        cardinality_ratio = B / N

        Q, beta_range, obj_scores = build_qubo(
            mu, cov, B=B, lambda_alpha=lambda_alpha,
            cardinality_ratio=cardinality_ratio,
            garch_vols=garch_vols,
        )

        try:
            selected, energy, violated, _ = solve_qubo_sa(
                Q, num_reads=1000, B=B, seed=i,
                beta_range=beta_range, n_seeds=3,
                num_sweeps=1000, obj_scores=obj_scores,
            )
        except Exception as e:
            logger.error(f"SA failed on {rebalance_date.date()}: {e}", exc_info=True)
            raise

        weights        = optimize_weights(mu, cov, selected,
                                          max_weight=MARKET_MAX_WEIGHT.get(market, 0.40),
                                          min_diversification_ratio=min_diversification_ratio)
        current_value  = np.dot(shares, prices.iloc[i].values) + cash
        current_prices = prices.iloc[i].values
        new_shares     = np.nan_to_num(
            (weights * current_value) / current_prices, nan=0.0, posinf=0.0
        )
        turnover = np.sum(np.abs(new_shares - shares) * current_prices)
        tc       = tc_rate * turnover
        cash     = current_value - np.dot(new_shares, current_prices) - tc
        shares   = new_shares

        selected_tickers = [tickers[j] for j in range(len(tickers)) if selected[j] == 1]
        rebalance_info.append({
            "date":             rebalance_date,
            "selected":         selected_tickers,
            "weights":          weights[selected == 1],
            "turnover":         turnover,
            "transaction_cost": tc,
            "cash":             cash,
        })
        logger.info(f"Selected {len(selected_tickers)} assets, "
                    f"turnover={turnover:,.0f}, tc={tc:,.0f}, cash={cash:,.0f}")

        next_i = min(i + rebalance_days, len(prices))
        for j in range(i, next_i):
            pv = np.dot(shares, prices.iloc[j].values) + cash
            portfolio_values.append(pv)
            dates.append(prices.index[j])

    if not portfolio_values:
        raise ValueError(f"No rebalancing periods executed for {market} -- "
                         f"check lookback_days={lookback_days} vs data length={len(prices)}")

    logger.info(f"Rebalancing complete -- final value: {portfolio_values[-1]:,.0f}, "
                f"periods: {len(portfolio_values)}")
    return pd.Series(portfolio_values, index=dates), rebalance_info

# Walk-Forward Backtest

In [102]:
def run_single_experiment(market, prices_all, volume_all, universe_label, universe_size,
                          B, lambda_alpha, lookback, backtest_days,
                          sa_seeds=5, num_reads=1000, num_sweeps=1000):
    """
    True walk-forward experiment: slides a training window across the backtest period
    and rebalances every rebalance_days steps with no look-ahead.
    """
    try:
        candidate_pool = build_universe(prices_all, volume_all,
                                        size=universe_size, lookback=lookback,
                                        market=market, pool_multiplier=2)
        prices_sub = candidate_pool
    except Exception as e:
        logger.error(f"Universe construction failed: {e}", exc_info=True)
        return {}

    tickers   = prices_sub.columns.tolist()
    pool_size = len(tickers)

    if len(prices_sub) < lookback + backtest_days:
        logger.warning(f"Insufficient data: {len(prices_sub)} < {lookback + backtest_days}")
        return {}

    rebalance_days = MARKET_REBALANCE_DAYS.get(market, 63)
    max_weight     = MARKET_MAX_WEIGHT.get(market, 0.40)
    rfr            = RISK_FREE_RATES.get(market, 0.0)
    min_dr         = MIN_DIVERSIFICATION_RATIO.get(market, 1.0)

    backtest_start   = len(prices_sub) - backtest_days
    all_port_returns = []
    all_dates        = []
    rebalance_log    = []

    current_weights  = None
    last_mu          = None
    last_cov         = None
    last_lambda_used = lambda_alpha  # TODO: capture dynamic effective lambda from build_qubo

    for i in range(backtest_start, len(prices_sub) - 1):
        steps_into_backtest = i - backtest_start
        is_rebalance        = (steps_into_backtest % rebalance_days == 0)

        if is_rebalance:
            train_prices = prices_sub.iloc[i - lookback: i]
            try:
                result     = compute_statistics(train_prices, use_garch=True, use_shrinkage=True)
                garch_vols = result[3] if len(result) == 4 else None
                mu, cov, _ = result[:3]
            except Exception as e:
                logger.warning(f"Statistics failed at step {i}: {e}")
                if current_weights is None:
                    continue
            else:
                try:
                    cardinality_ratio = B / pool_size

                    Q, beta_range, obj_scores = build_qubo(
                        mu, cov, B=B, lambda_alpha=lambda_alpha,
                        cardinality_ratio=cardinality_ratio,
                        garch_vols=garch_vols,
                    )
                    stability = run_stability_analysis(
                        Q, B, n_seeds=sa_seeds,
                        num_reads=num_reads, num_sweeps=num_sweeps,
                        beta_range=beta_range, obj_scores=obj_scores,
                    )
                    selected        = stability["consensus_solution"]
                    current_weights = optimize_weights(mu, cov, selected,
                                                       max_weight=max_weight,
                                                       min_diversification_ratio=min_dr)
                    last_mu  = mu
                    last_cov = cov

                    selected_tickers = [tickers[j] for j in np.where(selected == 1)[0]]
                    rebalance_log.append({
                        "date":     prices_sub.index[i],
                        "selected": selected_tickers,
                        "weights":  current_weights[selected == 1],
                        "stability": stability,
                    })
                    logger.debug(f"Rebalance @ {prices_sub.index[i].date()}: {selected_tickers}")

                except Exception as e:
                    logger.warning(f"QUBO failed at step {i}: {e}")
                    if current_weights is None:
                        continue

        if current_weights is None:
            continue

        today_prices    = prices_sub.iloc[i].values
        tomorrow_prices = prices_sub.iloc[i + 1].values
        daily_returns   = (tomorrow_prices - today_prices) / today_prices
        port_return     = float(np.dot(current_weights, daily_returns))
        all_port_returns.append(port_return)
        all_dates.append(prices_sub.index[i + 1])

    if len(all_port_returns) < 20:
        logger.warning(f"Too few walk-forward steps: {len(all_port_returns)}")
        return {}

    port_returns = np.array(all_port_returns)
    cum          = np.cumprod(1 + port_returns)
    total_return = float(cum[-1] - 1)
    n            = len(port_returns)
    ann_return   = float((1 + total_return) ** (252.0 / n) - 1) if n > 0 else 0.0
    cum_max      = np.maximum.accumulate(cum)
    max_drawdown = float(((cum - cum_max) / cum_max).min())
    rfr_adj      = ann_return - rfr

    ex_ante_sharpe = 0.0
    if last_mu is not None and last_cov is not None and current_weights is not None:
        port_ret  = float(current_weights @ last_mu)
        port_vol  = float(np.sqrt(max(current_weights @ last_cov @ current_weights, 1e-20)))
        ex_ante_sharpe = (port_ret - rfr) / port_vol if port_vol > 1e-10 else 0.0

    # Compute diversification ratio inline
    last_weights = current_weights
    if last_cov is not None and last_weights is not None:
        ind_vols  = np.sqrt(np.diag(last_cov))
        port_vol  = float(np.sqrt(max(last_weights @ last_cov @ last_weights, 1e-20)))
        div_ratio = float(last_weights @ ind_vols) / port_vol if port_vol > 1e-10 else 0.0
    else:
        div_ratio = 0.0

    # Pull stability metrics from the last rebalance
    last_stability = rebalance_log[-1]["stability"] if rebalance_log else {}

    last_selected = rebalance_log[-1]["selected"] if rebalance_log else []
    last_exp_ret  = float(last_weights @ last_mu)  if last_mu  is not None and last_weights is not None else 0.0
    last_vol      = float(np.sqrt(max(last_weights @ last_cov @ last_weights, 1e-20))) if last_cov is not None and last_weights is not None else 0.0

    logger.info(f"Walk-forward complete: {len(rebalance_log)} rebalances, "
                f"ex_ante_sharpe={ex_ante_sharpe:.4f}, ann_return={ann_return:.4f}")

    return {
        "market":                market,
        "universe_label":        universe_label,
        "universe_size":         pool_size,
        "B":                     B,
        "lambda_alpha":          lambda_alpha,
        "lambda_value":          last_lambda_used,  # TODO: capture dynamic effective_alpha * objective_scale
        "lookback":              lookback,
        "backtest_days":         backtest_days,
        "n_rebalances":          len(rebalance_log),
        "selected_assets":       last_selected,
        "num_selected":          B,
        "sharpe_ratio":          ex_ante_sharpe,
        "ann_return":            ann_return,
        "rfr_adj_ann_return":    rfr_adj,
        "total_return":          total_return,
        "max_drawdown":          max_drawdown,
        "sa_violation_rate":     last_stability.get("violation_rate", 0.0),
        "asset_consistency":     last_stability.get("asset_consistency", 1.0),
        "mean_energy":           last_stability.get("mean_energy", 0.0),
        "std_energy":            last_stability.get("std_energy", 0.0),
        "ex_ante_sharpe":        ex_ante_sharpe,
        "expected_return":       last_exp_ret,
        "volatility":            last_vol,
        "diversification_ratio": div_ratio,
    }


# Grid Search & Reporting

In [103]:
# _print_summary has been merged into print_final_report.
# This stub is retained to avoid NameError if called directly.
def _print_summary(df):
    print_final_report(df)


In [104]:
def run_experiment_grid(markets, universe_sizes, cardinalities,
                        lambda_alphas, lookbacks, start_date, end_date,
                        backtest_days=252, cache_dir="data_cache",
                        results_dir="results", num_reads=1000,
                        num_sweeps=1000, sa_seeds=5,
                        price_data=None, volume_data=None):
    os.makedirs(results_dir, exist_ok=True)
    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = os.path.join(results_dir, f"experiment_results_{timestamp}.csv")
    checkpoint  = os.path.join(results_dir, f"checkpoint_{timestamp}.csv")
    all_results   = []
    success_count = 0
    error_count   = 0
    first_write   = True

    for market in markets:
        logger.info(f"Processing market: {market}")
        try:
            if price_data and volume_data and market in price_data and market in volume_data:
                prices_all = price_data[market]
                volume_all = volume_data[market]
                logger.info(f"{market}: using pre-downloaded data "
                            f"({len(prices_all.columns)} assets, {len(prices_all)} periods)")
            else:
                prices_all, volume_all = download_data(market, start_date, end_date, cache_dir)
        except Exception as e:
            logger.error(f"Download failed for {market}: {e}", exc_info=True)
            continue

        market_success = 0
        market_errors  = 0

        for u_label, u_size in universe_sizes.items():
            seen_B = set()  # deduplicate B values within this universe
            for ratio, lambda_alpha, lookback in itertools.product(
                cardinalities, lambda_alphas, lookbacks
            ):
                B = max(1, int(ratio * u_size))

                # FIX 1: only skip if B would equal or exceed the full universe
                # removed the `B / u_size >= 0.4` guard that was killing cardinality=0.4
                if B >= u_size:
                    logger.debug(f"Skipping B={B}: equals or exceeds universe size {u_size}")
                    continue

                # skip duplicate B values for same universe (different ratios rounding to same B)
                b_key = (u_label, B, lambda_alpha, lookback)
                if b_key in seen_B:
                    logger.debug(f"Skipping duplicate: {b_key}")
                    continue
                seen_B.add(b_key)

                tag = f"{market}|{u_label}|B={B}|a={lambda_alpha}|lb={lookback}"
                logger.info(f"Running: {tag}")
                result = None
                try:
                    result = run_single_experiment(
                        market, prices_all, volume_all,
                        u_label, u_size, B,
                        lambda_alpha, lookback, backtest_days,
                        sa_seeds=sa_seeds, num_reads=num_reads, num_sweeps=num_sweeps
                    )
                    if result:
                        all_results.append(result)
                        success_count += 1; market_success += 1
                        logger.info(f"SUCCESS: {tag}")
                        pd.DataFrame([result]).to_csv(
                            checkpoint, mode='a', header=first_write, index=False
                        )
                        first_write = False
                    else:
                        error_count += 1; market_errors += 1
                        logger.warning(f"Empty result: {tag}")
                except Exception as e:
                    error_count += 1; market_errors += 1
                    logger.error(f"FAILED: {tag}", exc_info=True)
                finally:
                    del result
                    gc.collect()

        logger.info(f"{market} complete -- {market_success} success, {market_errors} errors")
        del prices_all, volume_all
        gc.collect()

    total      = success_count + error_count
    success_rt = 100 * success_count / total if total > 0 else 0
    logger.info(f"Grid complete: {success_count} successes, {error_count} errors "
                f"({success_rt:.1f}% success rate)")

    df = pd.DataFrame(all_results)
    del all_results; gc.collect()

    if df.empty:
        logger.warning("No results generated")
        return df

    df.to_csv(output_file, index=False)
    logger.info(f"Results saved to {output_file} ({len(df)} rows)")
    print_final_report(df)
    return df

# Plotting

In [105]:
BENCHMARK_LABELS = {
    "india":  "NIFTY 50",
    "usa":    "S&P 500",
    "europe": "EURO STOXX 50",
    "japan":  "Nikkei 225",
    "crypto": "BTC",
}

CYAN   = "#5bc8c8"
PINK   = "#c85b8a"
ORANGE = "#e89c45"
NAVY   = "#1a3a5c"
GREEN  = "#4caf7d"


def _save_plot(fig, name="plot", plots_dir="plots/experiment"):
    os.makedirs(plots_dir, exist_ok=True)
    fig.savefig(os.path.join(plots_dir, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info(f"Saved -> {name}.png")


def _style_ax(ax, title):
    ax.set_facecolor("white")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#cccccc")
    ax.tick_params(colors="#555555", labelsize=8)
    ax.grid(True, alpha=0.25, linestyle="--", color="#cccccc")
    ax.set_title(title, fontweight="bold", fontsize=10, pad=8, color="#222222")
    ax.xaxis.label.set_color("#555555")
    ax.yaxis.label.set_color("#555555")


def _millions_fmt(x, _):
    if abs(x) >= 1e6:   return f"{x/1e6:.2f}M"
    elif abs(x) >= 1e3: return f"{x/1e3:.1f}K"
    return f"{x:.2f}"


def _clean_ticker(t):
    for sfx in [".NS", ".T", ".L", ".AS", ".PA", ".SW", ".DE",
                ".MI", ".MC", ".CO", ".HE", ".ST", "-USD"]:
        t = t.replace(sfx, "")
    return t


In [106]:
def _rerun_best(best_row, prices_all, volume_all, bench_series,
                market, initial_capital, backtest_days):

    lookback       = int(best_row["lookback"])
    B              = int(best_row["B"])
    lambda_alpha   = float(best_row["lambda_alpha"])
    universe_label = best_row["universe_label"]
    universe_size  = int(best_row["universe_size"])

    try:
        candidate_pool = build_universe(
            prices_all, volume_all,
            size=universe_size, lookback=lookback,
            market=market, pool_multiplier=2,
        )
    except Exception as e:
        raise ValueError(f"Universe build failed in _rerun_best: {e}")

    # FIX 3: guard — rebalance_portfolio needs at least lookback + one rebalance period
    rebalance_days = MARKET_REBALANCE_DAYS.get(market, 63)
    min_required   = lookback + rebalance_days
    if len(candidate_pool) < min_required:
        raise ValueError(
            f"Insufficient data for plot: lookback={lookback} needs {min_required} rows, "
            f"have {len(candidate_pool)} -- skipping plot"
        )

    tickers = candidate_pool.columns.tolist()

    portfolio_values, rebalance_info = rebalance_portfolio(
        candidate_pool, tickers,
        market=market,
        initial_capital=initial_capital,
        B=B,
        lookback_days=lookback,
        lambda_alpha=lambda_alpha,
        min_diversification_ratio=MIN_DIVERSIFICATION_RATIO.get(market, 1.0),
    )

    weights       = rebalance_info[-1]["weights"] if rebalance_info else np.array([])
    bench_aligned = bench_series.reindex(portfolio_values.index).ffill()
    bench_values  = bench_aligned / bench_aligned.iloc[0] * initial_capital

    portfolio_rebalanced = pd.Series(
        [info["date"] for info in rebalance_info],
        name="rebalance_dates"
    )

    return portfolio_values, bench_values, portfolio_rebalanced, tickers, weights, rebalance_info

In [107]:
def plot_results(portfolio_values, benchmark_values, portfolio_rebalanced,
                 tickers, weights, rebalance_info,
                 plots_dir="plots/experiment",
                 market="", criterion="", meta=None):

    bench_label   = BENCHMARK_LABELS.get(market, "Benchmark")
    criterion_tag = f"Best {criterion.replace('_', ' ').title()}" if criterion else ""

    fig = plt.figure(figsize=(20, 16))
    fig.patch.set_facecolor("#f4f6f8")
    ax1 = plt.subplot(3, 2, 1); ax2 = plt.subplot(3, 2, 2)
    ax3 = plt.subplot(3, 2, 3); ax4 = plt.subplot(3, 2, 4)
    ax5 = plt.subplot(3, 2, 5); ax6 = plt.subplot(3, 2, 6)

    _style_ax(ax1, "Portfolio Performance vs Benchmark")
    ax1.plot(portfolio_values.index, portfolio_values.values,
             color=CYAN, linewidth=1.8, label="Strategy (No Rebalancing)")
    ax1.plot(benchmark_values.index, benchmark_values.values,
             color=PINK, linewidth=1.8, label=f"Benchmark ({bench_label})")
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(_millions_fmt))
    ax1.set_xlabel("Date"); ax1.set_ylabel("Value")
    ax1.legend(fontsize=8, framealpha=0.8)

    _style_ax(ax2, "Cumulative Returns")
    port_ret  = (portfolio_values / portfolio_values.iloc[0] - 1) * 100
    bench_ret = (benchmark_values / benchmark_values.iloc[0] - 1) * 100
    ax2.plot(port_ret.index,  port_ret.values,  color=CYAN, linewidth=1.8, label="Strategy")
    ax2.plot(bench_ret.index, bench_ret.values, color=PINK, linewidth=1.8, label="Benchmark")
    ax2.axhline(0, color="#999999", linewidth=0.8)
    ax2.set_xlabel("Date"); ax2.set_ylabel("Return (%)")
    ax2.legend(fontsize=8, framealpha=0.8)

    _style_ax(ax3, "Impact of Rebalancing")
    ax3.plot(portfolio_values.index, portfolio_values.values,
             color=CYAN,   linewidth=1.6, label="Without Rebalancing", alpha=0.85)
    ax3.plot(portfolio_rebalanced.index, portfolio_rebalanced.values,
             color=ORANGE, linewidth=2.0, label="With Rebalancing")
    ax3.plot(benchmark_values.index, benchmark_values.values,
             color=PINK,   linewidth=1.4, linestyle="--", alpha=0.6, label="Benchmark")
    for info in rebalance_info:
        ax3.axvline(info["date"], color="#aaaaaa", linestyle=":", alpha=0.4, linewidth=0.8)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(_millions_fmt))
    ax3.set_xlabel("Date"); ax3.set_ylabel("Value")
    ax3.legend(fontsize=8, framealpha=0.8)

    _style_ax(ax4, "Optimal Portfolio Weights")
    idx = np.where(weights > 0)[0]
    if len(idx):
        w = weights[idx]
        t = [_clean_ticker(tickers[i]) for i in idx]
        pairs = sorted(zip(w, t))
        w_sorted, t_sorted = zip(*pairs)
        bar_colors = ["#4caf7d","#e05c5c","#e89c45","#9b6fb5","#5bc8c8",
                      "#3a7abf","#c85b8a","#7dbf5b","#bf9b3a","#5b8ac8"][:len(t_sorted)]
        bars = ax4.barh(t_sorted, w_sorted, color=bar_colors, height=0.6, edgecolor="none")
        ax4.set_xlabel("Weight"); ax4.set_xlim(0, max(w_sorted) * 1.28)
        for bar, val in zip(bars, w_sorted):
            ax4.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                     f"{val:.3f} ({val:.1%})", va="center", ha="left",
                     fontsize=8, fontweight="bold", color="#333333")
        ax4.tick_params(axis="y", labelsize=9)

    _style_ax(ax5, f"Rolling Sharpe (window={MARKET_FR_WINDOW.get(market, 21)}d, Rebalanced)")
    if len(portfolio_rebalanced) > MARKET_FR_WINDOW.get(market, 21) + 5:
        window        = MARKET_FR_WINDOW.get(market, 21)
        rfr_daily     = RISK_FREE_RATES.get(market, 0.0) / 252
        port_daily    = portfolio_rebalanced.pct_change().dropna()
        excess        = port_daily - rfr_daily
        rolling_mean  = excess.rolling(window).mean()
        rolling_std   = excess.rolling(window).std(ddof=1)
        rolling       = (rolling_mean / rolling_std * np.sqrt(252)).dropna()
        ax5.plot(rolling.index, rolling.values, color=GREEN, linewidth=1.6)
        ax5.axhline(0, color="#999999", linewidth=0.8)
        ax5.fill_between(rolling.index, rolling.values, 0,
                         where=(rolling.values >= 0), alpha=0.15, color=GREEN)
        ax5.fill_between(rolling.index, rolling.values, 0,
                         where=(rolling.values < 0),  alpha=0.15, color="#e05c5c")
    else:
        ax5.text(0.5, 0.5, "Insufficient data for rolling Sharpe",
                 ha="center", va="center", transform=ax5.transAxes,
                 color="#888888", fontsize=9)
    ax5.set_xlabel("Date"); ax5.set_ylabel("Sharpe Ratio")

    _style_ax(ax6, "Asset Holdings per Rebalance Period")
    if rebalance_info:
        clean = [_clean_ticker(t) for t in tickers]
        held  = np.zeros((len(tickers), len(rebalance_info)))
        for col, info in enumerate(rebalance_info):
            for t in info["selected"]:
                tc = _clean_ticker(t)
                if tc in clean:
                    held[clean.index(tc), col] = 1
        row_mask  = held.sum(axis=1) > 0
        held_filt = held[row_mask]
        labs_filt = [clean[i] for i in range(len(clean)) if row_mask[i]]
        cmap = LinearSegmentedColormap.from_list("holdings", ["#e8eef5", NAVY], N=2)
        ax6.imshow(held_filt, aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
        ax6.set_yticks(range(len(labs_filt))); ax6.set_yticklabels(labs_filt, fontsize=8)
        ax6.set_xticks(range(len(rebalance_info)))
        ax6.set_xticklabels([info["date"].strftime("%Y-%m") for info in rebalance_info],
                            rotation=30, ha="right", fontsize=7)
        ax6.tick_params(axis="both", which="both", length=0)
        for x in np.arange(-0.5, len(rebalance_info), 1):
            ax6.axvline(x, color="white", linewidth=0.8)
        for y in np.arange(-0.5, len(labs_filt), 1):
            ax6.axhline(y, color="white", linewidth=0.8)

    sup = f"Portfolio Optimisation -- {market.upper()} | {criterion_tag}"
    if meta:
        sup += "\n" + " | ".join(f"{k}={v}" for k, v in meta.items())
    plt.suptitle(sup, fontweight="bold", fontsize=12, color="#1a1a2e", y=1.02)
    plt.tight_layout()
    _save_plot(fig, f"best_{criterion}_{market}", plots_dir)



def generate_all_plots(df, plots_dir="plots/experiment",
                       price_data=None, volume_data=None, bench_prices=None,
                       start_date="2024-01-01", end_date="2026-01-01",
                       cache_dir="data_cache", backtest_days=252,
                       initial_capital=1_600_000):
    if df.empty:
        logger.warning("Empty results DataFrame -- no plots generated")
        return

    markets = df["market"].unique().tolist()

    if price_data is None or volume_data is None:
        price_data, volume_data = {}, {}
        for m in markets:
            try:
                price_data[m], volume_data[m] = download_data(m, start_date, end_date, cache_dir)
            except Exception as e:
                logger.warning(f"Price download failed for {m}: {e}")

    if bench_prices is None:
        bench_prices = {}
        for m in markets:
            try:
                bench_prices[m] = download_benchmark_data(m, start_date, end_date, cache_dir)
            except Exception as e:
                logger.warning(f"Benchmark download failed for {m}: {e}")

    for market in markets:
        mdf          = df[df["market"] == market]
        prices_all   = price_data.get(market)
        volume_all   = volume_data.get(market)
        bench_series = bench_prices.get(market)
        if prices_all is None or volume_all is None or bench_series is None:
            logger.warning(f"[{market}] Missing data -- skipping plots"); continue

        for criterion in ["sharpe_ratio", "ann_return"]:
            if criterion not in mdf.columns or mdf[criterion].isna().all():
                logger.warning(f"[{market}] '{criterion}' empty -- skipping"); continue

            best_row = mdf.loc[mdf[criterion].idxmax()]
            meta = {"univ": best_row["universe_label"], "B": int(best_row["B"]),
                    "lb": int(best_row["lookback"]),
                    "lambda": round(best_row["lambda_alpha"], 1)}
            logger.info(f"[{market.upper()}] Best {criterion}: "
                        f"sharpe={best_row['sharpe_ratio']:.4f} | "
                        f"ann_ret={best_row['ann_return']:.2%} | params={meta}")
            try:
                (portfolio_values, benchmark_values, portfolio_rebalanced,
                 tickers, weights, rebalance_info) = _rerun_best(
                    best_row, prices_all, volume_all, bench_series,
                    market=market, initial_capital=initial_capital, backtest_days=backtest_days,
                )
                plot_results(
                    portfolio_values, benchmark_values, portfolio_rebalanced,
                    tickers, weights, rebalance_info,
                    plots_dir=plots_dir, market=market, criterion=criterion, meta=meta,
                )
            # replace the single except block with:
            except ValueError as e:
                logger.warning(f"[{market}] {criterion} plot skipped: {e}")
            except Exception as e:
                logger.error(f"[{market}] {criterion} plot failed: {e}", exc_info=True)

In [108]:
def print_final_report(df: pd.DataFrame):
    if df.empty:
        logger.warning("No results to report"); return
    sep   = "=" * 72
    lines = [
        f"\n{sep}", "EXPERIMENT SUMMARY", sep,
        f"Total runs:  {len(df)}",
        f"Markets:     {sorted(df['market'].unique().tolist())}",
        f"Universes:   {sorted(df['universe_label'].unique().tolist())}",
        f"B values:    {sorted(df['B'].unique().tolist())}",
        "",
        "VIOLATION RATE BY MARKET",
        *[f"  {m:<12} {r:.1%}"
          for m, r in df.groupby("market")["sa_violation_rate"].mean().items()],
        "",
        "MEAN RFR-ADJUSTED RETURN BY MARKET (cross-market comparable)",
        *[f"  {m:<12} {r:.2%}"
          for m, r in df.groupby("market")["rfr_adj_ann_return"].mean().items()],
        "",
        "MEAN SHARPE (EX-ANTE) BY MARKET",
        *[f"  {m:<12} {s:.4f}"
          for m, s in df.groupby("market")["sharpe_ratio"]
          .mean().sort_values(ascending=False).items()],
        "",
        "MEAN SHARPE BY UNIVERSE SIZE",
        *[f"  {u:<10} {s:.4f}"
          for u, s in df.groupby("universe_label")["sharpe_ratio"].mean().items()],
        "",
        "MEAN REBALANCES BY MARKET",
        *[f"  {m:<12} {g['n_rebalances'].mean():.1f} rebalances"
          for m, g in df.groupby("market")],
        "",
        "BEST CONFIGURATION PER MARKET (by ex-ante Sharpe)",
    ]
    for market, mdf in df.groupby("market"):
        best = mdf.loc[mdf["sharpe_ratio"].idxmax()]
        lines.append(
            f"  {market:<12} sharpe={best['sharpe_ratio']:.4f} | "
            f"B={int(best['B'])} | "
            f"lb={int(best['lookback'])} | alpha={best['lambda_alpha']} | "
            f"rfr_adj={best['rfr_adj_ann_return']:.2%}"
        )
    lines.append(f"\n{sep}\n")
    logger.info("\n".join(lines))

# Main

In [109]:
def main():
    import gc

    price_data  = {}
    volume_data = {}
    bench_data  = {}

    for m in MARKETS:
        try:
            price_data[m], volume_data[m] = download_data(m, START_DATE, END_DATE)
            bench_data[m]                 = download_benchmark_data(m, START_DATE, END_DATE)
        except Exception as e:
            logger.error(f"Data download failed for {m}: {e}", exc_info=True)

    df = run_experiment_grid(
        MARKETS, UNIVERSE_SIZES, CARDINALITIES,
        LAMBDA_ALPHAS, LOOKBACKS, START_DATE, END_DATE,
        backtest_days=BACKTEST_DAYS, num_reads=NUM_READS,
        num_sweeps=NUM_SWEEPS, sa_seeds=SA_SEEDS,
        price_data=price_data, volume_data=volume_data
    )
    gc.collect()

    if not df.empty:
        generate_all_plots(
            df, plots_dir=PLOTS_DIR,
            price_data=price_data, volume_data=volume_data,
            bench_prices=bench_data,
            start_date=START_DATE, end_date=END_DATE,
            backtest_days=BACKTEST_DAYS,
        )
        print_final_report(df)


In [ ]:
main()

15:16:23 - INFO - Loaded india from cache: (988, 50)
15:16:23 - INFO - Loaded benchmark ^NSEI from cache: 988 rows
15:16:23 - INFO - Processing market: india
15:16:23 - INFO - india: using pre-downloaded data (50 assets, 988 periods)
15:16:23 - INFO - Running: india|u40|B=8|a=0.5|lb=126
15:16:23 - INFO - Building universe: market=india, size=40, pool_size=80, lookback=126, max_vol=75%
15:16:23 - INFO - Trimmed to 40 assets by ADTV from 40 candidates
15:16:23 - INFO - Universe built: 40 assets -> ['TATASTEEL.NS', 'HDFCBANK.NS', 'KOTAKBANK.NS', 'ITC.NS', 'POWERGRID.NS', 'RELIANCE.NS', 'ICICIBANK.NS', 'SBIN.NS', 'WIPRO.NS', 'NTPC.NS', 'ONGC.NS', 'INFY.NS', 'BAJFINANCE.NS', 'BHARTIARTL.NS', 'AXISBANK.NS', 'COALINDIA.NS', 'HINDALCO.NS', 'INDUSINDBK.NS', 'TCS.NS', 'HCLTECH.NS', 'HDFCLIFE.NS', 'M&M.NS', 'SUNPHARMA.NS', 'ADANIPORTS.NS', 'TECHM.NS', 'JSWSTEEL.NS', 'DRREDDY.NS', 'LT.NS', 'HINDUNILVR.NS', 'CIPLA.NS', 'NESTLEIND.NS', 'ASIANPAINT.NS', 'TATACONSUM.NS', 'ADANIENT.NS', 'BAJAJFINSV.NS'

Error in callback <function _draw_all_if_interactive at 0x0000016942841440> (for post_execute), with arguments args (),kwargs {}:


ValueError: Date ordinal 3146720.14207496 converts to 10585-06-05T03:24:35.276544 (using epoch 1970-01-01T00:00:00), but Matplotlib dates must be between year 0001 and 9999.

ValueError: Date ordinal 3146720.14207496 converts to 10585-06-05T03:24:35.276544 (using epoch 1970-01-01T00:00:00), but Matplotlib dates must be between year 0001 and 9999.

<Figure size 2000x1600 with 6 Axes>

: 